# 从原始AF3输出聚合指标到summary CSV

读取每个complex的 metrics_summary.csv + confidences.json + summary_confidences.json，
聚合为每个方法的综合 summary CSV。

In [1]:
import os
import json
import numpy as np
import pandas as pd
from Bio.PDB import PDBParser, FastMMCIFParser, Superimposer

In [2]:
BASE = '/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/predict/original_result'
METHODS = ['msa_notemplate', 'msa_template', 'nomsa_notemplate', 'nomsa_template']

AVAILABLE = [m for m in METHODS if os.path.isdir(f'{BASE}/{m}/output') and len(os.listdir(f'{BASE}/{m}/output')) > 0]
print(f'Available methods with data: {AVAILABLE}')

Available methods with data: ['msa_notemplate', 'msa_template', 'nomsa_notemplate', 'nomsa_template']


In [ ]:
def extract_metrics_from_complex(base_dir, pdb):
    """从单个complex的AF3输出中提取所有置信度指标"""
    metrics_csv = os.path.join(base_dir, pdb, 'metrics_summary.csv')
    if not os.path.exists(metrics_csv):
        return None
    
    df = pd.read_csv(metrics_csv)
    if df.empty:
        return None
    
    rows = []
    for _, row in df.iterrows():
        seed, sample_id = int(row['seed']), int(row['id'])
        sample_dir = os.path.join(base_dir, pdb, f'seed-{seed}_sample-{sample_id}')
        
        # 读取summary_confidences
        summary_path = os.path.join(base_dir, pdb, f'{pdb}_seed-{seed}_sample-{sample_id}_summary_confidences.json')
        if not os.path.exists(summary_path):
            summary_path = os.path.join(sample_dir, f'{pdb}_seed-{seed}_sample-{sample_id}_summary_confidences.json')
        if not os.path.exists(summary_path):
            continue
        
        with open(summary_path) as f:
            summary = json.load(f)
        
        # 读取confidences (含原子pLDDT和PAE)
        conf_path = os.path.join(base_dir, pdb, f'{pdb}_seed-{seed}_sample-{sample_id}_confidences.json')
        if not os.path.exists(conf_path):
            conf_path = os.path.join(sample_dir, f'{pdb}_seed-{seed}_sample-{sample_id}_confidences.json')
        if not os.path.exists(conf_path):
            continue
        
        with open(conf_path) as f:
            conf = json.load(f)
        
        # ---- 计算各指标 ----
        
        # pep_plDDT (已从metrics_summary获取)
        pep_plddt = row['pep_plddt']
        
        # plDDT (所有原子的平均pLDDT)
        atom_plddts = np.array(conf['atom_plddts'])
        plddt = round(float(atom_plddts.mean()), 4)
        
        # ipTM / pTM / ranking_score
        iptm = round(float(summary['iptm']), 4)
        ptm = round(float(summary['ptm']), 4)
        ranking_score = round(float(summary['ranking_score']), 4)
        
        # pep_pTM (chain_ptm的第二个值对应肽链)
        chain_ptm = summary.get('chain_ptm', [0, 0])
        pep_ptm = round(float(chain_ptm[1]), 4) if len(chain_ptm) > 1 else 0.0
        
        # PAE相关指标
        pae = np.array(conf['pae'])  # shape (n_tokens, n_tokens)
        token_chain_ids = conf['token_chain_ids']
        n_tokens = len(token_chain_ids)
        
        # pAE_min: 全局PAE最小值
        pae_min = round(float(pae.min()), 4)
        
        # ipAE: 链间PAE (inter-chain PAE)
        # 取chain A(蛋白)到chain B(肽)的PAE
        chain_a_mask = np.array([c == 'A' for c in token_chain_ids])
        chain_b_mask = np.array([c == 'B' for c in token_chain_ids])
        inter_pae = pae[np.ix_(chain_a_mask, chain_b_mask)]  # A->B
        if inter_pae.size > 0:
            ipae = round(float(inter_pae.mean()), 4)
            ipsae_max = round(float(inter_pae.min(axis=1).max()), 4)  # per-res A min, then max
            ipsae_min = round(float(inter_pae.min(axis=1).min()), 4)
        else:
            ipae, ipsae_max, ipsae_min = 0.0, 0.0, 0.0
        
        # lRMSD (scRMSD = 肽链骨架RMSD)
        lrmsd = round(float(row['scRMSD']), 4)
        
        rows.append({
            'native': pdb,
            'seed': seed,
            'sample_id': sample_id,
            'pep_plDDT': pep_plddt,
            'plDDT': plddt,
            'ipTM': iptm,
            'pTM': ptm,
            'pep_pTM': pep_ptm,
            'ranking_score': ranking_score,
            'pAE_min': pae_min,
            'ipAE': ipae,
            'ipSAE_max': ipsae_max,
            'ipSAE_min': ipsae_min,
            'lRMSD': lrmsd,
        })
    
    return pd.DataFrame(rows) if rows else None


def aggregate_method(method_name):
    """聚合一个方法下所有complex的指标"""
    base_dir = os.path.join(BASE, method_name, 'output')
    if not os.path.isdir(base_dir):
        return None
    
    complexes = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    all_rows = []
    for pdb in sorted(complexes):
        result = extract_metrics_from_complex(base_dir, pdb)
        if result is not None:
            all_rows.append(result)
    
    if not all_rows:
        return None
    return pd.concat(all_rows, ignore_index=True)

In [ ]:
os.makedirs('./summary', exist_ok=True)

for method in AVAILABLE:
    # 将方法名映射到notebook中使用的命名
    if method == 'msa_template':
        name = 'pro_msa_template-pep_nomsa_notemplate'
    elif method == 'msa_notemplate':
        name = 'pro_msa_notemplate-pep_nomsa_notemplate'
    elif method == 'nomsa_template':
        name = 'pro_nomsa_template-pep_nomsa_notemplate'
    elif method == 'nomsa_notemplate':
        name = 'pro_nomsa_notemplate-pep_nomsa_notemplate'
    else:
        name = method
    
    result = aggregate_method(method)
    if result is not None:
        result.to_csv(f'./summary/{name}.csv', index=False)
        print(f'{name}: {len(result)} rows, {result["native"].nunique()} complexes')
    else:
        print(f'{name}: no data')

In [ ]:
# 验证生成的summary文件
for f in sorted(os.listdir('./summary')):
    df = pd.read_csv(f'./summary/{f}')
    print(f'{f}: shape={df.shape}, columns={list(df.columns)}')